In [1]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do numpy
def df_to_numpy(df, sloupce, od=0, do=None, dtype=float):
    """
    sloupce: jmeno sloupce (str) nebo seznam jmen (list[str])
    od, do:  rozsah radku podle pozice (od vcetne, do bez); do=None znamená až do konce
    """
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    data = df[sloupce].iloc[od:do].to_numpy(dtype=dtype)
    return data[:, 0] if len(sloupce) == 1 else data

# ufloat pro tuple
def tuple_unc_A(*velicina):
    list_velicin = [*velicina]
    return tuple(uf(np.mean(v), unc_A(v)) for v in list_velicin)

# array do latex
def array_to_latex(*array, dig=4, format='f', style='L'):
    for u in array:
        print(f'${u:.{dig}{format}{style}}$')

In [ ]:
# nacteni google tabulek
sheet_id = "SPREADSHEET_ID"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(url)